# MMAF-CD v25 — Stage G: explanation engine v2 + clinical decision-support report (+ F3n nuclei)

Builds what the thesis promised, but only from parts that **passed** Stages F/F2:

| block | what it does | evidence it is based on |
|---|---|---|
| **Evidence map** | per-organ best faithful method (Grad-CAM or Score-CAM) | F2 deletion/insertion + sanity (IG and RISE excluded: failed sanity) |
| **Per-image heatmap check** | deletes the top-20 % pixels vs random 20 % → *consistent / weak / not supported / inconclusive* | turns F2's population test into a per-case check |
| **Lesion location** | EigenCAM / Score-CAM + **MedSAM outline only where validated** (skin, thyroid) | F2 localisation (box→MedSAM Dice 0.67 / 0.54; kidney & oesophagus not validated) |
| **Uncertainty & referral** | calibrated probability, flip/rotate disagreement, conformal prediction set (90 %) — thresholds fitted on the **validation** split | F2 trust AUROC (TTA 0.85 > focus 0.44) |
| **Nuclei (H&E organs)** | official pretrained HoVer-Net (TIAToolbox) counts + types, labelled *exploratory* | F3n PQ table computed in this notebook |
| **Clinical card** | finding, category, review tier, meaning, confirmatory next step, **which staging system applies and what it needs** | curated table — never derives a stage from an image, never names a regimen |
| **Report** | printable HTML + FHIR R4 `DiagnosticReport` (validated with `fhir.resources` R4B) | — |
| **Evaluation** | on the test split: review rate, accuracy of accepted cases, silent errors, set coverage, heatmap-check rates | — |

**Inputs:** same as Stage F2 **plus** PanNuke (`andrewmvd/cancer-inst-segmentation-and-classification`) and MoNuSeg (`tuanledinh/monuseg2018`) for the nuclei table.
Optional: attach your **Stage F2 output** (`f2_results.json`) — the per-organ method choice is then read from it.
Accelerator GPU T4 ×2 · **Internet ON** (TIAToolbox weights, MedSAM, pip) · ~1–1.5 h · output `mmaf_v25_G.zip`.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# G0 — packages (installed BEFORE anything is imported, so nothing already loaded can clash)
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys
for pkg in ["tiatoolbox", "fhir.resources"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(f"pip {pkg}: {'ok' if r.returncode == 0 else 'FAILED — ' + r.stderr[-300:]}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# G-0 — standalone setup (shared with F2) (no old models, no old gateway, no 25-min base pipeline)
#   • finds the v25 table + 16 specialists in the attached mmaf-v25-e2 / e2p datasets
#   • re-maps image paths if Kaggle mounted a dataset at a different path than in the E2 run
#   • rebuilds the pancreas crop rows from msd-pancreas-crops (same split as E2P/F)
#   • rebuilds KiTS23 PNGs on demand (E2/F wrote them to /kaggle/working at run time)
#   • every image is checked lazily → a dataset that is missing just shrinks the sample, it never crashes
# ════════════════════════════════════════════════════════════════════════════════
import os, re, gc, glob, json, math, time, copy, random, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2, timm, torch, torch.nn as nn
from PIL import Image
warnings.filterwarnings("ignore"); Image.MAX_IMAGE_PIXELS = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T0 = time.time(); TIME_BUDGET_H = 11.0
def e_elapsed_h(): return (time.time() - T0) / 3600
def free_mem(tag=""):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"device={DEVICE}  gpus={torch.cuda.device_count()}  torch={torch.__version__}  timm={timm.__version__}")

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXT_SPLIT = (0.60, 0.10, 0.30)
E2_BACKBONE = "convnext_small.fb_in22k_ft_in1k"
IMN_MEAN = np.array([0.485, 0.456, 0.406], np.float32); IMN_STD = np.array([0.229, 0.224, 0.225], np.float32)
MALIGNANT_CLASSES = {
    "blood": {"early", "pre", "pro"}, "lymphoma": {"cll", "fl", "mcl"},
    "skin": {"actinic_keratosis", "basal_cell_carcinoma", "melanoma", "squamous_cell_carcinoma"},
    "cervical": {"dyk", "koc", "mep", "pab"}, "colon": {"adenocarcinoma"}, "brain_2d": {"glioma", "meningioma", "tumor"},
    "esophageal": {"malignant"}, "kidney": {"tumor"}, "lung": {"malignant"}, "oral": {"scc"}, "pancreatic": {"tumor"},
    "thyroid": {"malignant"}, "breast": {"malignant"}, "bladder": {"muscle_invasive", "non_muscle_invasive"},
    "gastric": {"tumor", "deb", "lym", "str"},
    "ovarian": {"clear_cell", "endometrioid", "high_grade_serous", "low_grade_serous", "mucinous"}}

# ---- image helpers (identical to Stage E/F) -----------------------------------------------------
def load_rgb(p, max_side=768):
    p = ensure(p); im = Image.open(p)
    try: im.draft("RGB", (max_side, max_side))
    except Exception: pass
    if im.mode in ("I;16", "I;16B", "I"):
        a = np.asarray(im, dtype=np.float32); a = (a - a.min()) / (a.max() - a.min() + 1e-6) * 255
        im = Image.fromarray(a.astype(np.uint8))
    a = np.asarray(im.convert("RGB")); s = max(a.shape[:2])
    if s > max_side:
        f = max_side / s; a = cv2.resize(a, (max(1, int(a.shape[1] * f)), max(1, int(a.shape[0] * f))), interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(a)
def to_tensor(img):
    x = (img.astype(np.float32) / 255. - IMN_MEAN) / IMN_STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))
def eval_img(img, size=224): return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
def list_images(root): return sorted(str(p) for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT and p.is_file())
def _u01(s): return int(hashlib.md5(s.encode()).hexdigest()[:12], 16) / float(16 ** 12)
def _assign(source, group, fr):
    u = _u01(f"{source}|{group}"); return "train" if u < fr[0] else ("val" if u < fr[0] + fr[1] else "test")

# ---- attached inputs: search only small inputs, never walk the big image datasets ----------------
_BIG = ("multi-cancer", "cbis-ddsm", "gastric-cancer-data", "cancer-inst-segmentation", "kits23", "isic2018", "leukemia",
        "pmram", "lunghist700", "crc-val", "tn3k", "ddti", "edd2020", "pancreasct", "skin-cancer", "mendeley-lbc",
        "ovarian-cancer", "monuseg", "/models/")
def _roots(big=False):
    out = []
    for pat in ("/kaggle/input/datasets/*/*", "/kaggle/input/notebooks/*/*", "/kaggle/input/*"):
        for r in glob.glob(pat):
            if Path(r).name in ("datasets", "notebooks", "models", "competitions") or not Path(r).is_dir(): continue
            if not big and any(b in r.lower() for b in _BIG): continue
            out.append(r)
    return sorted(set(out))
def find_all(pattern, maxdepth=5):
    hits = []
    for r in _roots():
        for d in range(0, maxdepth + 1): hits += glob.glob(r + "/*" * d + "/" + pattern)
    return sorted(set(hits))
def find_slug(slug):
    for pat in (f"/kaggle/input/datasets/*/{slug}", f"/kaggle/input/{slug}", f"/kaggle/input/notebooks/*/{slug}"):
        h = [x for x in glob.glob(pat) if Path(x).is_dir()]
        if h: return h[0]
    return None
print("attached inputs:"); [print("   ", r) for r in _roots(big=True)]

# ---- table ------------------------------------------------------------------------------------
_tabs = find_all("e_table.csv")
assert _tabs, "❌ e_table.csv not found — attach the mmaf-v25-e2 dataset"
_n = {t: sum(1 for _ in open(t, errors="ignore")) for t in _tabs}
E_PREV_TABLE = max(_n, key=_n.get)
TAB = pd.read_csv(E_PREV_TABLE, usecols=["path", "organ", "source", "kind", "group", "split", "label_set", "spec_ok", "holdout"],
                  dtype={"label_set": str, "group": str}, low_memory=False)
TAB["label_set"] = TAB.label_set.fillna(""); TAB["spec_ok"] = TAB.spec_ok.astype(str).str.lower().eq("true")
print(f"table: {E_PREV_TABLE}  ({len(TAB):,} rows; candidates: {[ (t, _n[t]) for t in _tabs ]})")

# ---- re-map dataset roots whose mount path changed --------------------------------------------
def _root_of(p):
    q = p.split("/")
    if p.startswith(("/kaggle/input/datasets/", "/kaggle/input/notebooks/")) and len(q) > 5: return "/".join(q[:6]), q[5]
    if p.startswith("/kaggle/input/") and len(q) > 3: return "/".join(q[:4]), q[3]
    return None, None
_rt = TAB.path.map(lambda p: _root_of(p)[0]); _moved = {}
for r in _rt.dropna().unique():
    if Path(r).exists(): continue
    new = find_slug(r.rstrip("/").split("/")[-1])
    if new and new != r:
        m = _rt == r; TAB.loc[m, "path"] = new + TAB.loc[m, "path"].str[len(r):]; _moved[r] = new
for a, b in _moved.items(): print(f"   re-mapped {a} → {b}")
_missing_roots = sorted(r for r in _rt.dropna().unique() if not Path(_moved.get(r, r)).exists())
for r in _missing_roots: print(f"   ⚠️ not attached: {r}  ({int((_rt == r).sum()):,} rows will be skipped)")

# ---- pancreas: crops from Stage P1b v2 (as E2P / F) -------------------------------------------
def find_msd_crops():
    for m in find_all("metadata.csv", 4):
        q = Path(m).parent
        if "crop" in q.name.lower() and (q / "tumor").is_dir() and (q / "normal").is_dir(): return q
    return None
_cr = find_msd_crops()
if _cr is not None:
    meta = pd.read_csv(_cr / "metadata.csv"); rows = []
    for src, name in [("msd", "msd_task07"), ("nih", "nih")]:
        d = meta[meta.source == src]
        for p, y, c in zip(d.path, d.label, d.case.astype(str)):
            rows.append({"path": str(_cr / p), "organ": "pancreatic", "source": name, "kind": "external", "group": c,
                         "split": "test" if src == "nih" else _assign(name, c, EXT_SPLIT), "label_set": y, "spec_ok": True, "holdout": src == "nih"})
    TAB = pd.concat([TAB[~((TAB.organ == "pancreatic") & TAB.source.isin(["msd_task07", "nih"]))], pd.DataFrame(rows)], ignore_index=True)
    print(f"pancreas crops: {_cr}  → {len(rows)} rows ({sum(r['split'] == 'test' for r in rows)} test)")
else:
    print("⚠️ msd-pancreas-crops not attached → pancreatic uses the whole-slice rows (the E2P specialist expects crops)")

# ---- KiTS23: PNGs + tumour masks rebuilt on demand --------------------------------------------
EXT_ROOT = Path("/kaggle/working/external"); (EXT_ROOT / "kits_png").mkdir(parents=True, exist_ok=True)
_KITS = find_slug("kits23-2d-kidney-tumor-segmentation"); KITS_IMG = KITS_MSK = None
if _KITS:
    for d in range(0, 4):                                   # literal last component → only directory names are listed
        if KITS_IMG is None: KITS_IMG = next((Path(q) for q in glob.glob(_KITS + "/*" * d + "/images") if Path(q).is_dir()), None)
        if KITS_MSK is None: KITS_MSK = next((Path(q) for q in glob.glob(_KITS + "/*" * d + "/masks") if Path(q).is_dir()), None)
print(f"KiTS23: images={KITS_IMG}  masks={KITS_MSK}")
def _kits_png(name):
    o = EXT_ROOT / "kits_png" / name
    if o.exists() or KITS_IMG is None or not (KITS_IMG / name).exists(): return o
    a = cv2.imread(str(KITS_IMG / name), cv2.IMREAD_UNCHANGED)
    a8 = (a / 257.).astype(np.uint8) if a.dtype == np.uint16 else a.astype(np.uint8)
    cv2.imwrite(str(o), cv2.cvtColor(a8, cv2.COLOR_GRAY2BGR) if a8.ndim == 2 else a8); return o
def _kits_mask(name):
    o = EXT_ROOT / "kits_png" / name.replace(".png", "_tumour.png")
    if not o.exists() and KITS_MSK is not None and (KITS_MSK / name).exists():
        mm = cv2.imread(str(KITS_MSK / name), cv2.IMREAD_UNCHANGED); cv2.imwrite(str(o), ((mm == 2) * 255).astype(np.uint8))
    return o
def ensure(p):
    p = str(p)
    if "/kits_png/" in p and not os.path.exists(p): return str(_kits_png(Path(p).name))
    return p
_EX = {}
def exists(p):
    if p not in _EX:
        try: _EX[p] = os.path.exists(ensure(p))
        except Exception: _EX[p] = False
    return _EX[p]
def first_existing(df, n):
    keep = []
    for i, p in zip(df.index, df.path):
        if exists(p): keep.append(i)
        if len(keep) >= n: break
    return df.loc[keep]
def per_group_existing(df, n, seed, keys=("organ", "source")):
    return pd.concat([first_existing(g, n) for _, g in df.sample(frac=1.0, random_state=seed).groupby(list(keys))] or [df.head(0)])

# ---- specialists (E2; pancreatic from E2P) ----------------------------------------------------
class SEHead(nn.Module):
    def __init__(self, dim, reduction=16):
        super().__init__(); b = max(1, dim // reduction)
        self.fc = nn.Sequential(nn.Linear(dim, b, bias=False), nn.GELU(), nn.Linear(b, dim, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)
class SpecialistModel(nn.Module):
    def __init__(self, backbone_name, n_classes, dropout=0.25, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0); D = self.backbone.num_features
        self.head = nn.Sequential(nn.Dropout(dropout), SEHead(D), nn.LayerNorm(D), nn.Dropout(dropout / 2), nn.Linear(D, n_classes))
    def forward(self, x): return self.head(self.backbone(x))

_cands = {}
for p in find_all("spec_v25/*.pth"): _cands.setdefault(Path(p).stem, []).append(p)
NEW_SPEC, E_CLASSES = {}, {}
for organ, ps in sorted(_cands.items()):
    best = None
    for p in ps:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        rank = (ck.get("stage", "E2") == "E2P", Path(p).stat().st_mtime)
        if best is None or rank > best[0]: best = (rank, p, ck)
    _, p, ck = best
    m = SpecialistModel(ck.get("backbone_name", E2_BACKBONE), len(ck["class_names"]), dropout=ck.get("dropout", 0.25))
    m.load_state_dict({k: v.float() for k, v in ck["model_state_dict"].items()}); m.eval()
    NEW_SPEC[organ] = {"model": m, "T": float(ck.get("temperature", 1.0)), "stage": ck.get("stage", "E2"), "path": p}
    E_CLASSES[organ] = list(ck["class_names"]); del ck
    print(f"  ✅ {organ:<11} {NEW_SPEC[organ]['stage']:<4} T={NEW_SPEC[organ]['T']:.2f}  {p}")
assert NEW_SPEC, "❌ no spec_v25/*.pth found — attach mmaf-v25-e2 (and mmaf-v25-e2p)"
E_ORGANS = sorted(NEW_SPEC); CLS_IDX = {o: {c: i for i, c in enumerate(E_CLASSES[o])} for o in E_ORGANS}
TAB = TAB[TAB.organ.isin(E_ORGANS)].reset_index(drop=True)
def _sets_of(rows, organ):
    return [",".join(str(CLS_IDX[organ][c]) for c in sorted(s.split("|")) if c in CLS_IDX[organ]) if s else "" for s in rows.label_set]

# ---- coverage report: which test images can actually be read ----------------------------------
print(f"\n{'organ':<11}{'source':<18}{'test rows':>10}{'readable (sample of 20)':>26}")
_cov = {}
for (o, s), g in TAB[TAB.split == "test"].groupby(["organ", "source"]):
    smp = g.sample(min(20, len(g)), random_state=0); r = float(np.mean([exists(p) for p in smp.path]))
    _cov[f"{o}|{s}"] = {"test_rows": int(len(g)), "readable_frac": r}
    print(f"{o:<11}{s:<18}{len(g):>10}{r:>20.0%}" + ("   ⚠️" if r < 0.5 else ""))

# ---- lesion masks for localisation ------------------------------------------------------------
def seg_pairs():
    out = {}
    isic = find_slug("isic2018-challenge-task1-data-segmentation")
    if isic:
        L = list_images(isic); gt = {Path(p).stem.replace("_segmentation", ""): p for p in L if Path(p).stem.endswith("_segmentation")}
        out["skin"] = [(p, gt[Path(p).stem]) for p in L if Path(p).stem in gt and not Path(p).stem.endswith("_segmentation")][:300]
    k = TAB[(TAB.organ == "kidney") & (TAB.source == "kits23") & (TAB.label_set == "tumor")]
    if len(k) and KITS_MSK is not None:
        pr = []
        for p in k.path:
            q = ensure(p)
            if os.path.exists(q):
                mk = _kits_mask(Path(q).name)
                if mk.exists(): pr.append((q, str(mk)))
            if len(pr) >= 300: break
        out["kidney"] = pr
    edd = find_slug("edd2020-endoscopy-detection-and-segmentation")
    if edd:
        rx = re.compile(r"^(.+)_(be|suspicious|hgd|cancer|polyp)(?:_mask)?$", re.I); mask_of, ims = {}, {}
        for p in list_images(edd):
            st = Path(p).stem; m = rx.match(st)
            if m and Path(p).suffix.lower() in (".tif", ".tiff", ".png"):
                if m.group(2).lower() in ("cancer", "hgd"): mask_of.setdefault(m.group(1).lower(), p)
            elif Path(p).suffix.lower() in (".jpg", ".jpeg", ".png"): ims[st.lower()] = p
        out["esophageal"] = [(ims[b], m) for b, m in mask_of.items() if b in ims][:300]
    tn = find_slug("tn3k-thyroid-nodule-region-segmentation-dataset")
    if tn:
        ri = [(p, Path(p).relative_to(tn).as_posix().lower()) for p in list_images(tn)]
        ims = {Path(p).stem: p for p, r in ri if r.split("/")[-2:-1] == ["test-image"]}
        gts = {Path(p).stem: p for p, r in ri if r.split("/")[-2:-1] == ["test-mask"]}
        out["thyroid"] = [(ims[k], gts[k]) for k in ims if k in gts][:300]
    for k, v in out.items(): print(f"   seg pairs {k}: {len(v)}")
    return out
print(f"\n✅ setup done in {time.time() - T0:.0f}s")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# G1 — clinical knowledge table (replaces the thesis CANCER_STAGING / CLINICAL_PROTOCOLS)
#   Rules this table follows, because the thesis version broke them:
#   • NO stage or grade is derived from an image. A single 2-D image cannot give TNM / FIGO / Lugano /
#     WHO-CNS grade; the report states WHICH system applies and WHAT information it needs.
#   • NO treatment regimens (the thesis listed hyper-CVAD, Stupp, VDLP …). Only the confirmatory next
#     step a clinician would order to verify an AI flag.
#   • Dataset classes are named for what they are (e.g. SIPaKMeD cell types are NOT CIN grades;
#     Early-pre-B / pre-B / pro-B are ALL subtypes, NOT stages; HMU-GC classes are tissue types).
#   • Every finding is marked by category: malignant · premalignant/abnormal · benign · normal · tissue.
#   • Urgency is a review-priority tier for the reading clinician, never "emergency".
# ════════════════════════════════════════════════════════════════════════════════
DISCLAIMER = ("Research prototype — not a medical device and not approved for clinical use. "
              "All findings are decision support only and must be confirmed by a qualified clinician.")
TIERS = {"malignant": "Priority specialist review", "abnormal": "Specialist review",
         "benign": "Routine", "normal": "Routine", "tissue": "Routine (tissue-type label)"}
C = lambda name, cat, meaning, nxt: {"name": name, "category": cat, "meaning": meaning, "next": nxt}

CLIN = {
 "blood": {"modality": "Peripheral-blood smear (microscopy)",
   "staging": "Acute lymphoblastic leukaemia is not TNM-staged. Risk group assignment uses age, white-cell count, cytogenetics/molecular genetics, CNS status and minimal residual disease after induction.",
   "classes": {
     "benign": C("Benign (non-blast) lymphoid cells", "benign", "No lymphoblast pattern detected by the model.", "Interpret with the full blood count and clinical context."),
     "early": C("Early pre-B ALL morphology pattern", "malignant", "Cell morphology resembles early pre-B lymphoblasts.", "Urgent haematology/haematopathology review. Diagnosis and B-lineage subtype require bone-marrow examination with flow-cytometric immunophenotyping."),
     "pre": C("Pre-B ALL morphology pattern", "malignant", "Cell morphology resembles pre-B lymphoblasts.", "Urgent haematology/haematopathology review. Diagnosis and B-lineage subtype require bone-marrow examination with flow-cytometric immunophenotyping."),
     "pro": C("Pro-B ALL morphology pattern", "malignant", "Cell morphology resembles pro-B lymphoblasts.", "Urgent haematology/haematopathology review. Diagnosis and B-lineage subtype require bone-marrow examination with flow-cytometric immunophenotyping.")}},
 "lymphoma": {"modality": "H&E histology",
   "staging": "Lymphomas are staged with the Lugano classification (Ann Arbor-based, PET-CT). CLL/SLL is staged with Rai or Binet using blood counts and examination.",
   "classes": {
     "cll": C("CLL / small lymphocytic lymphoma pattern", "malignant", "Morphology resembles CLL/SLL.", "Haematopathology review with immunohistochemistry/flow (e.g. CD5, CD23) to confirm subtype; staging work-up per Lugano or Rai/Binet."),
     "fl": C("Follicular lymphoma pattern", "malignant", "Morphology resembles follicular lymphoma.", "Haematopathology review with immunohistochemistry (e.g. CD10, BCL2) and grading; staging work-up per Lugano."),
     "mcl": C("Mantle cell lymphoma pattern", "malignant", "Morphology resembles mantle cell lymphoma.", "Haematopathology review with immunohistochemistry (e.g. cyclin D1, CD5); staging work-up per Lugano.")}},
 "skin": {"modality": "Dermoscopy / clinical photograph",
   "staging": "Melanoma: AJCC 8th edition (Breslow thickness, ulceration, nodes, metastasis) — requires excision histology. Cutaneous SCC: AJCC 8th edition (head & neck). BCC is usually not staged.",
   "classes": {
     "melanoma": C("Melanoma pattern", "malignant", "Lesion appearance resembles melanoma.", "Dermatology referral; excisional biopsy for histopathology."),
     "basal_cell_carcinoma": C("Basal cell carcinoma pattern", "malignant", "Lesion appearance resembles BCC.", "Dermatology referral; diagnostic biopsy as indicated."),
     "squamous_cell_carcinoma": C("Squamous cell carcinoma pattern", "malignant", "Lesion appearance resembles cutaneous SCC.", "Dermatology referral; diagnostic biopsy for histopathology."),
     "actinic_keratosis": C("Actinic keratosis pattern", "abnormal", "Appearance resembles actinic keratosis (premalignant keratinocyte lesion).", "Dermatology review; treatment or monitoring per clinician."),
     "nevus": C("Melanocytic naevus pattern", "benign", "Appearance resembles a benign naevus.", "Routine; re-evaluate if the lesion changes (size, shape, colour)."),
     "seborrheic_keratosis": C("Seborrhoeic keratosis pattern", "benign", "Appearance resembles seborrhoeic keratosis.", "Routine; re-evaluate if atypical or changing."),
     "normal": C("No lesion pattern", "normal", "No lesion pattern detected.", "Routine.")}},
 "cervical": {"modality": "Pap smear cytology (single-cell images)",
   "staging": "Cytology is not staged. Slide-level results are reported with The Bethesda System; cervical cancer is staged with FIGO 2018.",
   "classes": {
     "dyk": C("Dyskeratotic squamous cells", "abnormal", "Cell-level abnormality (dyskeratosis) — not a CIN grade.", "Cytopathologist review to assign the slide-level Bethesda category; management per ASCCP risk-based guidelines (e.g. HPV testing, colposcopy)."),
     "koc": C("Koilocytotic cells (HPV cytopathic effect)", "abnormal", "Cell-level HPV-associated change — not a CIN grade.", "Cytopathologist review to assign the slide-level Bethesda category; management per ASCCP risk-based guidelines."),
     "mep": C("Metaplastic squamous cells", "normal", "Normal cell type of the transformation zone.", "Routine; slide-level interpretation by cytology staff."),
     "pab": C("Parabasal cells", "normal", "Normal cell type (often atrophic/post-menopausal smears).", "Routine; slide-level interpretation by cytology staff."),
     "sfi": C("Superficial-intermediate squamous cells", "normal", "Normal cell type.", "Routine.")}},
 "colon": {"modality": "H&E histology",
   "staging": "Colorectal cancer: AJCC/UICC TNM 8th edition — requires the resection specimen (depth, nodes) and staging imaging.",
   "classes": {
     "adenocarcinoma": C("Adenocarcinoma pattern", "malignant", "Tissue resembles colorectal adenocarcinoma.", "Pathologist confirmation; if confirmed, mismatch-repair/MSI testing and multidisciplinary-team referral with staging CT."),
     "benign": C("Benign colonic tissue pattern", "benign", "No adenocarcinoma pattern detected.", "Routine; pathologist sign-out.")}},
 "brain_2d": {"modality": "Brain MRI (2-D slice)",
   "staging": "Brain tumours are graded, not TNM-staged: WHO CNS5 (2021) integrated histological and molecular grading (e.g. IDH, 1p/19q) requires tissue.",
   "classes": {
     "glioma": C("Glioma pattern", "malignant", "Appearance resembles a glioma (grade cannot be determined from imaging alone).", "Neuroradiology review with contrast-enhanced MRI; neurosurgical / neuro-oncology referral."),
     "meningioma": C("Meningioma pattern", "abnormal", "Appearance resembles a meningioma (most are WHO grade 1).", "Neuroradiology review; neurosurgical opinion."),
     "tumor": C("Other intracranial tumour pattern (incl. pituitary)", "abnormal", "Intracranial mass pattern not classified as glioma or meningioma.", "Neuroradiology review; specialist referral according to location.")}},
 "esophageal": {"modality": "Upper-GI endoscopy image",
   "staging": "Oesophageal cancer: AJCC 8th edition TNM — requires histology plus CT / endoscopic ultrasound (± PET-CT).",
   "classes": {
     "malignant": C("High-grade dysplasia / cancer pattern", "malignant", "Region resembles high-grade dysplasia or carcinoma.", "Targeted endoscopic biopsy for histology; if confirmed, staging with CT/EUS."),
     "benign": C("Barrett's oesophagus without high-grade features", "benign", "Barrett's-type mucosa without a high-grade pattern.", "Surveillance per gastroenterology guidelines (e.g. BSG/ACG).")}},
 "kidney": {"modality": "Abdominal CT (2-D slice)",
   "staging": "Renal cell carcinoma: AJCC 8th edition TNM on cross-sectional imaging; histology confirms subtype. Not every renal mass is malignant.",
   "classes": {
     "tumor": C("Renal mass", "abnormal", "A renal tumour region is suspected on this slice.", "Radiology review with multiphase contrast CT/MRI characterisation; urology referral."),
     "normal": C("No renal mass on this slice", "normal", "No tumour region detected on this slice.", "Routine; one slice does not exclude a lesion elsewhere.")}},
 "lung": {"modality": "H&E histology",
   "staging": "Lung cancer: AJCC/UICC TNM 8th edition — requires imaging (CT, PET-CT) and pathology.",
   "classes": {
     "malignant": C("Lung carcinoma pattern", "malignant", "Tissue resembles lung carcinoma.", "Pathologist confirmation with immunohistochemistry for subtype (e.g. TTF-1, p40); biomarker testing for non-small-cell carcinoma."),
     "benign": C("Benign lung tissue pattern", "benign", "No carcinoma pattern detected.", "Routine; pathologist sign-out.")}},
 "oral": {"modality": "H&E histology",
   "staging": "Oral cavity cancer: AJCC 8th edition TNM (includes depth of invasion).",
   "classes": {
     "scc": C("Oral squamous cell carcinoma pattern", "malignant", "Tissue resembles oral SCC.", "Pathologist confirmation; head-and-neck multidisciplinary-team referral."),
     "normal": C("Normal oral mucosa pattern", "normal", "No carcinoma pattern detected.", "Routine.")}},
 "pancreatic": {"modality": "Abdominal CT (pancreas-centred crop)",
   "staging": "Pancreatic cancer: AJCC 8th edition TNM; resectability assessed on pancreatic-protocol CT/MRI (e.g. NCCN criteria).",
   "classes": {
     "tumor": C("Pancreatic tumour pattern", "malignant", "Region resembles a pancreatic tumour.", "Radiology review with pancreatic-protocol CT/MRI; hepatopancreatobiliary MDT; tissue diagnosis (e.g. EUS-guided biopsy) as indicated."),
     "normal": C("No pancreatic tumour pattern", "normal", "No tumour pattern detected on this crop.", "Routine; one slice does not exclude a lesion.")}},
 "thyroid": {"modality": "Thyroid ultrasound",
   "staging": "Thyroid nodules are risk-stratified on ultrasound (ACR TI-RADS or EU-TIRADS); FNA cytology is reported with The Bethesda System; thyroid cancer is staged with AJCC 8th edition.",
   "classes": {
     "malignant": C("Suspicious thyroid nodule pattern", "malignant", "Nodule appearance resembles malignant nodules.", "Radiologist TI-RADS assessment; fine-needle aspiration if TI-RADS and size criteria are met."),
     "benign": C("Benign-appearing thyroid nodule pattern", "benign", "Nodule appearance resembles benign nodules.", "Follow-up per TI-RADS recommendations.")}},
 "breast": {"modality": "Mammography (region of interest)",
   "staging": "Breast imaging is reported with BI-RADS; breast cancer staging is AJCC 8th edition (anatomic + prognostic: grade, ER/PR/HER2).",
   "classes": {
     "malignant": C("Suspicious for malignancy", "malignant", "Region resembles malignant mammographic findings.", "Radiologist BI-RADS assessment; image-guided core biopsy if BI-RADS 4–5."),
     "benign": C("Benign-appearing finding", "benign", "Region resembles benign mammographic findings.", "Radiologist BI-RADS assessment; routine screening interval if confirmed benign.")}},
 "bladder": {"modality": "Bladder MRI",
   "staging": "Bladder cancer: TNM (T2 or higher = muscle-invasive) confirmed by TURBT histology with detrusor muscle; MRI is scored with VI-RADS.",
   "classes": {
     "muscle_invasive": C("Muscle-invasive pattern", "malignant", "Appearance suggests muscle-invasive disease.", "Radiologist VI-RADS assessment; urology/MDT; TURBT with detrusor-muscle sampling for confirmation."),
     "non_muscle_invasive": C("Non-muscle-invasive pattern", "malignant", "Appearance suggests non-muscle-invasive disease.", "Radiologist VI-RADS assessment; urology; TURBT and risk stratification (e.g. EAU).")}},
 "gastric": {"modality": "H&E histology (tissue-type tiles)",
   "staging": "Gastric cancer: AJCC 8th edition TNM — requires resection or endoscopic staging (EUS, CT).",
   "classes": {
     "tumor": C("Tumour epithelium", "malignant", "Tile resembles gastric carcinoma epithelium.", "Pathologist confirmation; HER2 and mismatch-repair/MSI testing on confirmed adenocarcinoma."),
     "normal": C("Normal gastric mucosa", "normal", "Normal mucosa tile.", "Routine."),
     "adi": C("Adipose tissue", "tissue", "Tissue-type label, not a diagnosis.", "None (tissue component)."),
     "deb": C("Debris / necrosis", "tissue", "Tissue-type label, not a diagnosis.", "None (tissue component)."),
     "lym": C("Lymphocytes", "tissue", "Tissue-type label, not a diagnosis.", "None (tissue component)."),
     "muc": C("Mucus", "tissue", "Tissue-type label, not a diagnosis.", "None (tissue component)."),
     "mus": C("Smooth muscle", "tissue", "Tissue-type label, not a diagnosis.", "None (tissue component)."),
     "str": C("Stroma", "tissue", "Tissue-type label, not a diagnosis.", "None (tissue component).")}},
 "ovarian": {"modality": "H&E histology",
   "staging": "Ovarian cancer: FIGO 2014 surgical staging.",
   "classes": {
     "high_grade_serous": C("High-grade serous carcinoma pattern", "malignant", "Tissue resembles HGSC.", "Gynaecological pathology confirmation with immunohistochemistry (e.g. WT1, p53); BRCA/HRD testing per guidelines; gynae-oncology MDT."),
     "low_grade_serous": C("Low-grade serous carcinoma pattern", "malignant", "Tissue resembles LGSC.", "Gynaecological pathology confirmation with immunohistochemistry; gynae-oncology MDT."),
     "clear_cell": C("Clear cell carcinoma pattern", "malignant", "Tissue resembles clear cell carcinoma.", "Gynaecological pathology confirmation with immunohistochemistry; gynae-oncology MDT."),
     "endometrioid": C("Endometrioid carcinoma pattern", "malignant", "Tissue resembles endometrioid carcinoma.", "Gynaecological pathology confirmation with immunohistochemistry (incl. mismatch repair); gynae-oncology MDT."),
     "mucinous": C("Mucinous carcinoma pattern", "malignant", "Tissue resembles mucinous carcinoma.", "Gynaecological pathology confirmation (exclude metastasis from GI tract); gynae-oncology MDT.")}},
}
HE_ORGANS = {"colon", "gastric", "lung", "lymphoma", "oral", "ovarian"}

# safety net: a class the table does not know gets a neutral entry + a warning (never a guessed diagnosis)
for o in E_ORGANS:
    CLIN.setdefault(o, {"modality": "unknown", "staging": "Not specified.", "classes": {}})
    for c in E_CLASSES[o]:
        if c not in CLIN[o]["classes"]:
            print(f"   ⚠️ clinical table has no entry for {o}/{c} — neutral text used")
            CLIN[o]["classes"][c] = C(c.replace("_", " "), "abnormal", "Model class without curated clinical text.", "Specialist review.")
print(f"✅ clinical table: {len(CLIN)} organs, {sum(len(v['classes']) for v in CLIN.values())} findings — no image-derived stage, no regimens")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# G2 — explanation engine v2, uncertainty, nuclei, MedSAM
# ════════════════════════════════════════════════════════════════════════════════
import base64, io, datetime, uuid, html as _html
import torch.nn.functional as _F
from torchvision.transforms import GaussianBlur
G_OUT = Path("/kaggle/working/g_out"); (G_OUT / "reports").mkdir(parents=True, exist_ok=True)
GR = {"meta": {"table": E_PREV_TABLE, "specialists": {o: {k: v for k, v in s.items() if k != "model"} for o, s in NEW_SPEC.items()},
               "coverage": _cov, "moved_roots": _moved}, "config": {}, "thresholds": {}, "eval": {}, "nuclei_pq": {}, "reports": []}
def g_save(): (G_OUT / "g_results.json").write_text(json.dumps(GR, indent=1, default=float))
_blur = GaussianBlur(31, 10.0); _AMP = torch.cuda.is_available()
MODEL_VERSION = "MMAF-CD v25 (E2/E2P specialists, ConvNeXt-S in22k)"

# ---- per-organ method choice: Stage F2 results (attached json overrides the built-in copy) --------------
EVIDENCE = {"bladder": "Score-CAM", "blood": "Score-CAM", "brain_2d": "Grad-CAM", "breast": "Score-CAM", "cervical": "Score-CAM",
            "colon": "Score-CAM", "esophageal": "Grad-CAM", "gastric": "Grad-CAM", "kidney": "Score-CAM", "lung": "Score-CAM",
            "lymphoma": "Score-CAM", "oral": "Score-CAM", "ovarian": "Grad-CAM", "pancreatic": "Grad-CAM", "skin": "Grad-CAM", "thyroid": "Grad-CAM"}
LOCALISER = {"skin": ("EigenCAM", 0.670, 0.497), "thyroid": ("Score-CAM", 0.538, 0.264),     # (method, box→MedSAM Dice, random-map Dice)
             "kidney": ("EigenCAM", 0.241, 0.052), "esophageal": ("Score-CAM", 0.311, 0.307)}
_f2 = find_all("f2_results.json")
if _f2:
    try:
        d = json.load(open(_f2[0]))
        EVIDENCE.update({o: m for o, m in d.get("best", {}).items() if m in ("Grad-CAM", "Score-CAM", "Grad-CAM++", "EigenCAM")})
        for o, v in d.get("loc", {}).items():
            ok = {m: x["medsam"] for m, x in v["methods"].items() if m not in ("random", "Integrated Gradients", "RISE") and x.get("medsam") is not None}
            if ok:
                m = max(ok, key=ok.get); LOCALISER[o] = (m, ok[m], v["methods"]["random"]["medsam"])
        print(f"✅ method choice read from {_f2[0]}")
    except Exception as e: print("   f2_results.json unreadable:", e)
def outline_validated(o):
    if o not in LOCALISER: return False
    _, d, r = LOCALISER[o]; return d >= 0.45 and d >= r + 0.10
GR["config"] = {"evidence_method": EVIDENCE, "localiser": LOCALISER, "outline_validated": {o: outline_validated(o) for o in LOCALISER}}
print("evidence map per organ:", EVIDENCE); print("lesion outline validated for:", [o for o in LOCALISER if outline_validated(o)])

# ---- explanation methods (hook-free; identical to F2) ----------------------------------------------------
def _feats(model, x):
    bb = model.backbone; h = bb.stem(x); outs = []
    for st in bb.stages: h = st(h); outs.append(h)
    return outs[-2], bb.norm_pre(h)
def _logits(model, h): return model.head(model.backbone.forward_head(h))
def _nz(t): return torch.where(torch.isfinite(t), t, torch.zeros_like(t))
def _prep(model):
    for p in model.parameters(): p.requires_grad_(False)
    return model.to(DEVICE).eval()
def _up(m, hw): return cv2.resize(np.asarray(m, np.float32), (hw[1], hw[0]), interpolation=cv2.INTER_LINEAR)
def _norm(m): m = np.asarray(m, np.float32); return (m - m.min()) / (m.max() - m.min() + 1e-8)

def cam_maps(model, x, t, want=("Grad-CAM", "Grad-CAM++", "EigenCAM", "Score-CAM")):
    with torch.enable_grad():
        xi = x.detach().clone().requires_grad_(True); s3, h = _feats(model, xi); h.retain_grad()
        _logits(model, h)[0, t].backward()
    A, g = h.detach()[0].float(), h.grad[0].float(); out = {}
    out["Grad-CAM"] = torch.relu((g.mean((1, 2))[:, None, None] * A).sum(0))
    gp2, gp3 = g ** 2, g ** 3
    a = _nz(torch.where(g != 0, gp2 / (2 * gp2 + A.sum((1, 2))[:, None, None] * gp3 + 1e-7), torch.zeros_like(g)))
    out["Grad-CAM++"] = torch.relu(((a * torch.relu(g)).sum((1, 2))[:, None, None] * A).sum(0))
    M = A.reshape(A.shape[0], -1).T; M = M - M.mean(0, keepdim=True)
    try:
        _, _, Vh = torch.linalg.svd(M, full_matrices=False); e = (M @ Vh[0]).reshape(A.shape[1:])
        if torch.corrcoef(torch.stack([e.flatten(), A.mean(0).flatten()]))[0, 1] < 0: e = -e
        out["EigenCAM"] = torch.relu(e)
    except Exception: out["EigenCAM"] = torch.zeros_like(A[0])
    if "Score-CAM" in want:
        with torch.no_grad():
            k = min(64, A.shape[0]); idx = A.mean((1, 2)).topk(k).indices
            Mk = _F.interpolate(A[idx][None], size=x.shape[2:], mode="bilinear", align_corners=False)[0]
            lo, hi = Mk.flatten(1).min(1)[0][:, None, None], Mk.flatten(1).max(1)[0][:, None, None]; Mk = (Mk - lo) / (hi - lo + 1e-8); ps = []
            for i in range(0, k, 32):
                with torch.cuda.amp.autocast(enabled=_AMP): ps.append(torch.softmax(model(x * Mk[i:i + 32, None]).float(), 1)[:, t])
            out["Score-CAM"] = torch.relu((torch.cat(ps)[:, None, None] * A[idx]).sum(0))
    return {k: _norm(_up(v.cpu().numpy(), x.shape[2:])) for k, v in out.items() if k in want}

@torch.no_grad()
def heatmap_check(model, x, t, cam, T, frac=0.20, n_rand=3):
    """per-image faithfulness: confidence drop when the top-`frac` pixels of the map are blurred vs random pixels."""
    H, W = x.shape[2:]; k = int(frac * H * W); base = _blur(x)
    ranks = [torch.as_tensor(np.argsort(-cam.ravel())[:k].copy(), device=x.device)] + [torch.randperm(H * W, device=x.device)[:k] for _ in range(n_rand)]
    xs = [x]
    for r in ranks:
        m = torch.zeros(H * W, device=x.device); m[r] = 1; m = m.view(1, 1, H, W); xs.append(x * (1 - m) + base * m)
    p = torch.softmax(model(torch.cat(xs)).float() / T, 1)[:, t].cpu().numpy()
    d_cam, d_rand = float(p[0] - p[1]), float(np.mean(p[0] - p[2:]))
    if max(abs(d_cam), abs(d_rand)) < 0.02: status = "inconclusive"
    elif d_cam >= 0.05 and d_cam >= d_rand + 0.05: status = "consistent"
    elif d_cam > 0 and d_cam > d_rand: status = "weak"
    else: status = "not supported"
    return {"status": status, "drop_highlighted": d_cam, "drop_random": d_rand, "p0": float(p[0])}

# ---- calibrated prediction + flip/rotate disagreement, batched --------------------------------------------
@torch.no_grad()
def predict_rows(organ, paths, bs=32):
    s = NEW_SPEC[organ]; model = _prep(s["model"]); P, TTA = [], []
    for i in range(0, len(paths), bs):
        x = torch.stack([to_tensor(eval_img(load_rgb(p))) for p in paths[i:i + bs]]).to(DEVICE)
        with torch.cuda.amp.autocast(enabled=_AMP):
            outs = [model(a).float() for a in (x, torch.flip(x, [3]), torch.flip(x, [2]), torch.rot90(x, 1, [2, 3]))]
        pr = [torch.softmax(o / s["T"], 1) for o in outs]
        P.append(pr[0].cpu()); TTA.append(torch.stack([0.5 * (q - pr[0]).abs().sum(1) for q in pr[1:]]).mean(0).cpu())
    s["model"] = model.cpu(); return torch.cat(P).numpy(), torch.cat(TTA).numpy()
def _true_sets(rows, organ):
    return [{int(c) for c in st.split(",")} if st else set() for st in _sets_of(rows, organ)]

# ---- thresholds + conformal q̂ fitted on the VALIDATION split (never on test) ---------------------------
ALPHA = 0.10; REVIEW_TARGET = 0.10
print(f"\n{'organ':<11}{'n val':>6}{'τ_tta':>8}{'τ_prob':>8}{'q̂(LAC)':>9}{'val review':>11}")
for organ in E_ORGANS:
    try:
        V = TAB[(TAB.organ == organ) & (TAB.split == "val") & TAB.spec_ok & (TAB.label_set != "")]
        V = per_group_existing(V, max(40, 400 // max(1, V.source.nunique())), 3).reset_index(drop=True) if len(V) else V
        if len(V) < 20: print(f"{organ:<11}{len(V):>6}  too few validation images → pooled defaults"); continue
        P, tta = predict_rows(organ, V.path.tolist()); msp = P.max(1); ts = _true_sets(V, organ)
        sc = np.array([1 - max(P[i, j] for j in s) for i, s in enumerate(ts)])
        n = len(sc); q = float(np.quantile(sc, min(1.0, math.ceil((n + 1) * (1 - ALPHA)) / n), method="higher"))
        tau_t, tau_p = float(np.quantile(tta, 1 - REVIEW_TARGET / 2)), float(np.quantile(msp, REVIEW_TARGET / 2))
        single = (P >= 1 - q).sum(1) == 1                       # same three rules as decide()
        rv = float(((tta > tau_t) | (msp < tau_p) | ~single).mean())
        GR["thresholds"][organ] = {"n_val": int(n), "tau_tta": tau_t, "tau_prob": tau_p, "qhat": q, "val_review_rate": rv}
        print(f"{organ:<11}{n:>6}{tau_t:>8.3f}{tau_p:>8.3f}{q:>9.3f}{rv:>11.3f}", flush=True)
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"   ⚠️ thresholds {organ}: {e}")
_pool = ({k: float(np.median([v[k] for v in GR["thresholds"].values()])) for k in ["tau_tta", "tau_prob", "qhat"]} if GR["thresholds"]
         else {"tau_tta": 0.10, "tau_prob": 0.50, "qhat": 0.50})      # last resort: fixed conservative values
for organ in E_ORGANS:
    if organ not in GR["thresholds"]:
        GR["thresholds"][organ] = {**_pool, "n_val": 0, "val_review_rate": None, "uncalibrated": True}
        print(f"   ⚠️ {organ}: thresholds NOT fitted on its own validation data — {'pooled from other organs' if len(GR['thresholds']) > 1 else 'fixed defaults'}")
g_save()

def decide(organ, p, tta):
    th = GR["thresholds"][organ]; cls = E_CLASSES[organ]
    pset = [cls[j] for j in np.argsort(-p) if p[j] >= 1 - th["qhat"]]
    reasons = []
    if tta > th["tau_tta"]: reasons.append("prediction changes when the image is flipped/rotated")
    if p.max() < th["tau_prob"]: reasons.append("calibrated confidence below the validated threshold")
    if not pset: reasons.append("no class reaches the 90 % conformal coverage bar")
    elif len(pset) > 1: reasons.append(f"{len(pset)} findings remain plausible ({', '.join(pset)})")
    return {"accept": not reasons, "reasons": reasons, "prediction_set": pset}

# ---- MedSAM (lesion outline, only where validated) ---------------------------------------------------------
MEDSAM = None
try:
    from transformers import SamModel, SamProcessor
    MEDSAM = (SamModel.from_pretrained("wanglab/medsam-vit-base").to(DEVICE).eval(), SamProcessor.from_pretrained("wanglab/medsam-vit-base"))
    print("✅ MedSAM loaded")
except Exception as e: print("   MedSAM unavailable:", type(e).__name__, str(e)[:120])
def mask_box(m, pad=0.10):
    ys, xs = np.where(m)
    if len(xs) == 0: return None
    h, w = m.shape; px, py = pad * (xs.max() - xs.min() + 1), pad * (ys.max() - ys.min() + 1)
    return [max(0, xs.min() - px), max(0, ys.min() - py), min(w - 1, xs.max() + px), min(h - 1, ys.max() + py)]
def largest(cm):
    n, lab, st, _ = cv2.connectedComponentsWithStats(cm.astype(np.uint8))
    return (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])) if n > 1 else cm.astype(bool)
@torch.no_grad()
def medsam_box(img_rgb, box):
    sam, proc = MEDSAM
    inp = proc(Image.fromarray(img_rgb), input_boxes=[[list(map(float, box))]], return_tensors="pt").to(DEVICE)
    out = sam(**inp, multimask_output=False)
    return proc.image_processor.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu(), inp["reshaped_input_sizes"].cpu())[0][0, 0].numpy().astype(bool)

# ---- official HoVer-Net (TIAToolbox, PanNuke weights) — direct model call, works on TIAToolbox 1.x and 2.x -----
HV = None; NUC_TYPES = {1: "neoplastic", 2: "inflammatory", 3: "connective", 4: "dead", 5: "non-neoplastic epithelial"}
try:
    import tiatoolbox
    from tiatoolbox.models.architecture import get_pretrained_model
    from tiatoolbox.models.architecture.hovernet import HoVerNet as _HVN
    try: _m, _ = get_pretrained_model("hovernet_fast-pannuke")
    except TypeError: _m, _ = get_pretrained_model(pretrained_model="hovernet_fast-pannuke")
    HV = getattr(_m, "module", _m).to(DEVICE).eval()
    print(f"✅ official HoVer-Net (TIAToolbox {tiatoolbox.__version__}, hovernet_fast-pannuke)")
except Exception as e: print("   official HoVer-Net unavailable:", type(e).__name__, str(e)[:160])

@torch.no_grad()
def hovernet_tile(img):
    """img: H×W×3 uint8 (≈0.25 µm/px expected). → instance map, per-instance type."""
    IN, OUT = 256, 164; P_ = (IN - OUT) // 2; H, W = img.shape[:2]
    pad = np.pad(img, ((P_, P_ + max(0, OUT - H)), (P_, P_ + max(0, OUT - W)), (0, 0)), mode="reflect")
    Hc, Wc = max(H, OUT), max(W, OUT)
    ys = sorted(set(list(range(0, Hc - OUT, OUT)) + [Hc - OUT])); xs = sorted(set(list(range(0, Wc - OUT, OUT)) + [Wc - OUT]))
    win = [(y, x) for y in ys for x in xs]; npm = np.zeros((Hc, Wc), np.float32); hvm = np.zeros((Hc, Wc, 2), np.float32); tpm = np.zeros((Hc, Wc), np.int32)
    for i in range(0, len(win), 16):
        ww = win[i:i + 16]
        o = HV(torch.from_numpy(np.stack([pad[y:y + IN, x:x + IN] for y, x in ww])).permute(0, 3, 1, 2).float().to(DEVICE))
        a = torch.softmax(o["np"], 1)[:, 1].cpu().numpy(); hv = o["hv"].permute(0, 2, 3, 1).cpu().numpy()
        tp = o["tp"].argmax(1).cpu().numpy() if "tp" in o else np.zeros_like(a, np.int32)
        for (y, x), a_, h_, t_ in zip(ww, a, hv, tp): npm[y:y + OUT, x:x + OUT] = a_; hvm[y:y + OUT, x:x + OUT] = h_; tpm[y:y + OUT, x:x + OUT] = t_
    inst = np.asarray(_HVN._proc_np_hv(npm[:H, :W, None], hvm[:H, :W])).astype(np.int32); tpm = tpm[:H, :W]
    types = {}
    for iid in np.unique(inst)[1:]:
        v = tpm[inst == iid]; v = v[v > 0]; types[int(iid)] = int(np.bincount(v).argmax()) if len(v) else 0
    return inst, types
def nuclei_summary(img):
    h, w = img.shape[:2]; s = min(h, w, 512); y0, x0 = (h - s) // 2, (w - s) // 2; crop = img[y0:y0 + s, x0:x0 + s]
    inst, types = hovernet_tile(crop); cnt = {NUC_TYPES.get(t, "unlabelled"): 0 for t in list(NUC_TYPES) + [0]}
    for t in types.values(): cnt[NUC_TYPES.get(t, "unlabelled")] += 1
    n = len(types); return {"n_nuclei": n, "counts": cnt, "neoplastic_fraction": (cnt["neoplastic"] / n) if n else None, "crop_px": s}, crop, inst, types
g_save(); free_mem("G2")


In [ ]:
try:
    # ════════════════════════════════════════════════════════════════════════════════
    # G3 — evaluate the full decision pipeline on the TEST split (what each report's "reliability" box quotes)
    # ════════════════════════════════════════════════════════════════════════════════
    N_EVAL, N_CHECK = 200, 40
    print(f"{'organ':<11}{'n':>5}{'review':>8}{'acc all':>9}{'acc kept':>9}{'silent err':>11}{'set cov':>9}{'set size':>9}{'map ok':>8}")
    for organ in E_ORGANS:
        if e_elapsed_h() > TIME_BUDGET_H - 1.5: print("⏱ time budget: evaluation stopped"); break
        try:
            Te = TAB[(TAB.organ == organ) & (TAB.split == "test") & TAB.spec_ok & (TAB.label_set != "")]
            Te = per_group_existing(Te, max(25, N_EVAL // max(1, Te.source.nunique())), 11).reset_index(drop=True) if len(Te) else Te
            if len(Te) < 10: print(f"{organ:<11} too few readable test images"); continue
            P, tta = predict_rows(organ, Te.path.tolist()); ts = _true_sets(Te, organ)
            dec = [decide(organ, P[i], tta[i]) for i in range(len(Te))]
            ok = np.array([int(P[i].argmax()) in ts[i] for i in range(len(Te))]); acc = np.array([d["accept"] for d in dec])
            cov = np.mean([any(E_CLASSES[organ].index(c) in ts[i] for c in d["prediction_set"]) for i, d in enumerate(dec)])
            size = np.mean([len(d["prediction_set"]) for d in dec])
            # per-image heatmap check on a subset
            model = _prep(NEW_SPEC[organ]["model"]); st = []
            for p in Te.path.head(N_CHECK):
                x = to_tensor(eval_img(load_rgb(p))).unsqueeze(0).to(DEVICE)
                with torch.no_grad(): t = int(model(x).argmax(1))
                meth = EVIDENCE.get(organ, "Grad-CAM"); cam = cam_maps(model, x, t, want=(meth,))[meth]
                st.append(heatmap_check(model, x, t, cam, NEW_SPEC[organ]["T"])["status"])
            NEW_SPEC[organ]["model"] = model.cpu()
            r = {"n": int(len(Te)), "sources": sorted(Te.source.unique().tolist()), "review_rate": float(1 - acc.mean()),
                 "acc_all": float(ok.mean()), "acc_accepted": float(ok[acc].mean()) if acc.any() else None,
                 "silent_error": float((acc & ~ok).mean()), "errors_caught": float((~acc & ~ok).sum() / max(1, (~ok).sum())),
                 "set_coverage": float(cov), "set_size": float(size),
                 "heatmap_check": {k: st.count(k) / len(st) for k in ["consistent", "weak", "not supported", "inconclusive"]} if st else {}}
            GR["eval"][organ] = r
            print(f"{organ:<11}{r['n']:>5}{r['review_rate']:>8.3f}{r['acc_all']:>9.3f}{(r['acc_accepted'] or float('nan')):>9.3f}"
                  f"{r['silent_error']:>11.3f}{r['set_coverage']:>9.3f}{r['set_size']:>9.2f}{r['heatmap_check'].get('consistent', 0):>8.2f}", flush=True)
        except Exception as e:
            import traceback; traceback.print_exc(); print(f"   ⚠️ eval {organ}: {e}")
        g_save(); free_mem(f"eval {organ}")
    if GR["eval"]:
        E_ = GR["eval"].values(); n = sum(v["n"] for v in E_)
        GR["eval_pooled"] = {k: float(sum(v[k] * v["n"] for v in E_ if v[k] is not None) / n) for k in ["review_rate", "acc_all", "silent_error", "set_coverage"]}
        print("pooled:", {k: round(v, 3) for k, v in GR["eval_pooled"].items()})
    g_save()

except Exception as _e:
    import traceback; traceback.print_exc()
    print('⚠️ G3 evaluation failed — skipped:', type(_e).__name__, _e)
    try: g_save()
    except Exception: pass


In [ ]:
try:
    # ════════════════════════════════════════════════════════════════════════════════
    # G4 — clinical decision-support reports (HTML for reading/printing + FHIR R4 DiagnosticReport JSON)
    # ════════════════════════════════════════════════════════════════════════════════
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    try:
        from fhir.resources.R4B.diagnosticreport import DiagnosticReport as _FHIR_DR
    except Exception as e: _FHIR_DR = None; print("   fhir.resources unavailable — JSON written but not schema-validated:", e)
    TYPE_COL = {1: (230, 40, 40), 2: (40, 200, 40), 3: (40, 90, 230), 4: (240, 200, 0), 5: (230, 120, 0), 0: (200, 200, 200)}

    def analyse(organ, path):
        """the full per-case pipeline used by the report."""
        s = NEW_SPEC[organ]; model = _prep(s["model"]); img = load_rgb(path); x = to_tensor(eval_img(img)).unsqueeze(0).to(DEVICE)
        P, tta = predict_rows(organ, [path]); p, tt = P[0], float(tta[0]); model = _prep(s["model"])
        t = int(p.argmax()); cls = E_CLASSES[organ][t]; card = CLIN[organ]["classes"][cls]; d = decide(organ, p, tt)
        ev = EVIDENCE.get(organ, "Grad-CAM"); loc = LOCALISER.get(organ, ("EigenCAM", None, None))[0]
        maps = cam_maps(model, x, t, want=tuple({ev, loc, "EigenCAM"}))
        chk = heatmap_check(model, x, t, maps[ev], s["T"])
        R = {"organ": organ, "path": path, "probs": {c: float(v) for c, v in zip(E_CLASSES[organ], p)}, "pred": cls, "p": float(p[t]),
             "tta": tt, "decision": d, "card": card, "evidence_method": ev, "heatmap_check": chk, "localiser": loc,
             "outline_validated": outline_validated(organ), "maps": maps, "img": img}
        if R["outline_validated"] and MEDSAM is not None:
            lm = _up(maps[loc], img.shape[:2]); b = mask_box(largest(lm >= 0.5))
            R["outline"] = medsam_box(img, b) if b else None
        if organ in HE_ORGANS and HV is not None:
            try: R["nuclei"], R["nuc_crop"], R["nuc_inst"], R["nuc_types"] = nuclei_summary(img)
            except Exception as e: R["nuclei_error"] = str(e)
        s["model"] = model.cpu(); return R

    def _png(fig):
        b = io.BytesIO(); fig.savefig(b, format="png", dpi=110, bbox_inches="tight"); plt.close(fig); return base64.b64encode(b.getvalue()).decode()

    def panels(R):
        img = R["img"]; im224 = eval_img(img); cols = [("Image", im224, None), (f"Evidence map ({R['evidence_method']})", im224, R["maps"][R["evidence_method"]])]
        cols.append((f"Attention region ({R['localiser']})" + ("" if R["organ"] in LOCALISER else " — not validated"), im224, R["maps"][R["localiser"]]))
        if R.get("outline") is not None: cols.append(("Lesion outline (MedSAM, validated)", img, R["outline"]))
        if "nuclei" in R: cols.append(("Nuclei (HoVer-Net, exploratory)", R["nuc_crop"], ("nuc", R["nuc_inst"], R["nuc_types"])))
        fig, ax = plt.subplots(1, len(cols), figsize=(3.1 * len(cols), 3.3))
        for a, (title, base, ov) in zip(np.atleast_1d(ax), cols):
            a.imshow(base); a.set_title(title, fontsize=8); a.axis("off")
            if ov is None: continue
            if isinstance(ov, tuple):
                _, inst, types = ov; over = base.copy()
                for iid, t in types.items():
                    cs, _ = cv2.findContours((inst == iid).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                    cv2.drawContours(over, cs, -1, TYPE_COL.get(t, (200, 200, 200)), 1)
                a.imshow(over)
            elif ov.dtype == bool:
                cs, _ = cv2.findContours(ov.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                o2 = base.copy(); cv2.drawContours(o2, cs, -1, (0, 255, 0), max(2, base.shape[0] // 150)); a.imshow(o2)
            else: a.imshow(ov, cmap="jet", alpha=0.45)
        plt.tight_layout(); return _png(fig)

    CHECK_TXT = {"consistent": "Heatmap consistent with the model's decision — blurring the highlighted 20 % of the image changed the confidence by {a} points vs {b} points for random regions.",
                 "weak": "Heatmap only weakly supported — blurring the highlighted region changed the confidence by {a} points vs {b} points for random regions. Interpret the map with caution.",
                 "not supported": "Heatmap NOT supported for this image — the highlighted regions mattered no more than random regions ({a} vs {b} points). Do not use the map to judge where the finding is.",
                 "inconclusive": "Heatmap check inconclusive — the confidence barely changes when parts of the image are removed ({a} vs {b} points)."}
    def _pts(drop): return f"{-100 * drop:+.0f}"

    def report(R, case_id):
        o = R["organ"]; card = R["card"]; d = R["decision"]; ev = GR["eval"].get(o, {}); th = GR["thresholds"][o]
        tier = TIERS[card["category"]] if d["accept"] else "Human review required — AI result not accepted"
        when = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"); chk = R["heatmap_check"]
        rel = (f"On {ev['n']} held-out test images for this organ: {ev['review_rate']:.0%} were sent to review; accepted cases were "
               f"{ev['acc_accepted']:.1%} correct; {ev['silent_error']:.1%} of all cases were wrong and not flagged. "
               f"The prediction set contained the true finding in {ev['set_coverage']:.0%} of cases.") if ev and ev.get("acc_accepted") is not None else "Organ-level test performance not available in this run."
        loc_txt = (f"Lesion outline shown: {R['localiser']} box → MedSAM, validated on lesion masks (Dice {LOCALISER[o][1]:.2f} vs {LOCALISER[o][2]:.2f} for a random map)."
                   if R.get("outline") is not None else
                   (f"Attention region only — localisation for this organ was tested and is NOT reliable (Dice {LOCALISER[o][1]:.2f}); no outline is drawn." if o in LOCALISER
                    else "Attention region only — no lesion masks were available to validate localisation for this organ; it is not a lesion outline."))
        nuc = R.get("nuclei"); nuc_txt = ""
        if nuc:
            parts = ", ".join(f"{k} {v}" for k, v in nuc["counts"].items() if v)
            nuc_txt = (f"<p><b>Nuclei (exploratory):</b> {nuc['n_nuclei']} nuclei in a {nuc['crop_px']} px centre crop ({parts}); "
                       f"neoplastic fraction {nuc['neoplastic_fraction']:.0%}." if nuc["neoplastic_fraction"] is not None else "") + \
                      " Pretrained HoVer-Net (PanNuke); image magnification is not calibrated, so counts are not clinical measurements.</p>"
        probs = "".join(f"<tr><td>{_html.escape(CLIN[o]['classes'][c]['name'])}</td><td>{v:.1%}</td></tr>" for c, v in sorted(R["probs"].items(), key=lambda kv: -kv[1]))
        pset = ", ".join(CLIN[o]["classes"][c]["name"] for c in d["prediction_set"]) or "— (empty)"
        reasons = "".join(f"<li>{_html.escape(r)}</li>" for r in d["reasons"])
        col = {"malignant": "#b3261e", "abnormal": "#b26a00", "benign": "#1e6b34", "normal": "#1e6b34", "tissue": "#555"}[card["category"]] if d["accept"] else "#6a1b9a"
        html_doc = f"""<!doctype html><html><head><meta charset="utf-8"><title>{case_id} — AI decision-support report</title>
    <style>body{{font-family:Arial,Helvetica,sans-serif;max-width:980px;margin:24px auto;color:#222;line-height:1.45}}
    h1{{font-size:20px;margin:0}} h2{{font-size:15px;border-bottom:1px solid #ccc;padding-bottom:3px;margin-top:22px}}
    .banner{{background:#fff4e5;border:1px solid #f0b46a;padding:8px 12px;font-size:13px}} .res{{border-left:6px solid {col};padding:8px 14px;background:#fafafa}}
    table{{border-collapse:collapse;font-size:13px}} td,th{{border:1px solid #ddd;padding:3px 8px}} .meta{{font-size:12px;color:#555}}
    img{{max-width:100%}} @media print{{.banner{{-webkit-print-color-adjust:exact}}}}</style></head><body>
    <h1>AI-assisted image analysis — decision-support report</h1>
    <p class="meta">Case {case_id} · {when} · {_html.escape(MODEL_VERSION)} · organ/specimen supplied with the request: <b>{o}</b> · modality: {_html.escape(CLIN[o]['modality'])} · no patient identifiers stored</p>
    <div class="banner"><b>Not for diagnostic use.</b> {_html.escape(DISCLAIMER)}</div>
    <h2>Result</h2><div class="res"><p style="font-size:17px;margin:4px 0"><b>{_html.escape(card['name'])}</b> &nbsp;·&nbsp; category: {card['category']} &nbsp;·&nbsp; calibrated probability {R['p']:.1%}</p>
    <p><b>Review tier:</b> {_html.escape(tier)}</p><p><b>Prediction set (90 % coverage):</b> {_html.escape(pset)}</p>
    {('<p><b>Why the AI result was not accepted:</b></p><ul>' + reasons + '</ul>') if not d['accept'] else '<p>Accepted by the reliability checks (calibrated confidence, flip/rotate stability, single-finding prediction set).</p>'}</div>
    <h2>Images and explanations</h2><img src="data:image/png;base64,{panels(R)}">
    <p><b>Evidence map ({R['evidence_method']}):</b> {CHECK_TXT[chk['status']].format(a=_pts(chk['drop_highlighted']), b=_pts(chk['drop_random']))}</p>
    <p><b>Location:</b> {_html.escape(loc_txt)}</p>{nuc_txt}
    <h2>Clinical interpretation</h2><p><b>What the finding means:</b> {_html.escape(card['meaning'])}</p>
    <p><b>Suggested confirmatory next step:</b> {_html.escape(card['next'])}</p>
    <p><b>Stage / grade:</b> not determinable from this image. {_html.escape(CLIN[o]['staging'])}</p>
    <h2>Reliability of this system for this organ</h2><p>{_html.escape(rel)}</p>
    <h2>All class probabilities</h2><table><tr><th>Finding</th><th>Calibrated probability</th></tr>{probs}</table>
    <h2>Technical details</h2><p class="meta">Specialist: {_html.escape(Path(NEW_SPEC[o]['path']).name)} ({NEW_SPEC[o]['stage']}), temperature T={NEW_SPEC[o]['T']:.2f}.
    {'⚠️ Thresholds NOT fitted on this organ (insufficient validation images) — treat the accept/review decision as uncalibrated. ' if th.get('uncalibrated') else ''}Thresholds fitted on the validation split (n={th.get('n_val')}): flip/rotate disagreement ≤ {th['tau_tta']:.3f}, calibrated confidence ≥ {th['tau_prob']:.3f}, LAC conformal q̂={th['qhat']:.3f} (α=0.10).
    Flip/rotate disagreement for this image: {R['tta']:.3f}. Explanation methods chosen from the Stage-F2 benchmark (faithfulness vs random, randomisation sanity test).</p>
    </body></html>"""
        obs = lambda i, text, **v: {"resourceType": "Observation", "id": f"o{i}", "status": "preliminary", "code": {"text": text}, **v}
        O = [obs(1, "AI finding", valueString=card["name"]),
             obs(2, "AI finding category", valueString=card["category"]),
             obs(3, "Calibrated probability of the finding", valueQuantity={"value": round(R["p"], 4), "unit": "probability"}),
             obs(4, "Conformal prediction set (90% coverage)", valueString=pset),
             obs(5, "AI result accepted by reliability checks", valueBoolean=bool(d["accept"])),
             obs(6, "Flip/rotate prediction disagreement", valueQuantity={"value": round(R["tta"], 4), "unit": "total variation"}),
             obs(7, f"Heatmap check ({R['evidence_method']})", valueString=chk["status"]),
             obs(8, "Review tier", valueString=tier)]
        if nuc: O.append(obs(9, "Nuclei count in centre crop (exploratory)", valueInteger=int(nuc["n_nuclei"])))
        fh = {"resourceType": "DiagnosticReport", "id": case_id.replace("_", "-"), "status": "preliminary",
              "category": [{"text": "AI-assisted image analysis (decision support, research prototype)"}],
              "code": {"text": f"AI image analysis — {o} ({CLIN[o]['modality']})"},
              "effectiveDateTime": datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z", "contained": O,
              "result": [{"reference": f"#{x['id']}"} for x in O],
              "conclusion": f"{card['name']} ({card['category']}), p={R['p']:.2f}. {tier}. Stage not determinable from image. {DISCLAIMER}",
              "presentedForm": [{"contentType": "text/html", "title": f"{case_id}.html", "data": base64.b64encode(html_doc.encode()).decode()}]}
        valid = None
        if _FHIR_DR is not None:
            try: (_FHIR_DR.model_validate(fh) if hasattr(_FHIR_DR, "model_validate") else _FHIR_DR.parse_obj(fh)); valid = True
            except Exception as e: valid = False; print(f"   FHIR validation failed for {case_id}: {str(e)[:200]}")
        (G_OUT / "reports" / f"{case_id}.html").write_text(html_doc); (G_OUT / "reports" / f"{case_id}.fhir.json").write_text(json.dumps(fh, indent=1))
        return {"case": case_id, "organ": o, "finding": card["name"], "category": card["category"], "p": R["p"], "accepted": bool(d["accept"]),
                "tier": tier, "heatmap_check": chk["status"], "outline": R.get("outline") is not None, "nuclei": nuc["n_nuclei"] if nuc else None,
                "fhir_valid": valid, "source": TAB.loc[TAB.path == R["path"], "source"].iloc[0] if (TAB.path == R["path"]).any() else "?"}

    PER_ORGAN = 2
    for organ in E_ORGANS:
        if e_elapsed_h() > TIME_BUDGET_H - 0.5: break
        try:
            Te = TAB[(TAB.organ == organ) & (TAB.split == "test") & TAB.spec_ok & (TAB.label_set != "")]
            Te = first_existing(Te.sample(frac=1, random_state=5), 12)
            if not len(Te): continue
            Rs = []
            for p in Te.path:
                try: Rs.append(analyse(organ, p))
                except Exception as e: print(f"   {organ}: {type(e).__name__}: {e}")
                if len(Rs) >= 6: break
            pick = []                                               # one accepted case + one review case when available
            for want in (True, False):
                c = [r for r in Rs if r["decision"]["accept"] == want and r not in pick]
                if c: pick.append(sorted(c, key=lambda r: r["card"]["category"] != "malignant")[0])
            for r in Rs:
                if len(pick) >= PER_ORGAN: break
                if r not in pick: pick.append(r)
            for i, R in enumerate(pick[:PER_ORGAN]):
                GR["reports"].append(report(R, f"{organ}-{i + 1:02d}"))
                print(f"  📄 {organ}-{i + 1:02d}  {R['card']['name']:<42} p={R['p']:.2f}  {'accepted' if R['decision']['accept'] else 'REVIEW'}  map={R['heatmap_check']['status']}", flush=True)
        except Exception as e:
            import traceback; traceback.print_exc(); print(f"   ⚠️ reports {organ}: {e}")
        g_save(); free_mem(f"reports {organ}")

    rows = "".join(f"<tr><td><a href='{r['case']}.html'>{r['case']}</a></td><td>{_html.escape(r['finding'])}</td><td>{r['category']}</td><td>{r['p']:.2f}</td>"
                   f"<td>{'yes' if r['accepted'] else '<b>review</b>'}</td><td>{r['heatmap_check']}</td><td>{'yes' if r['outline'] else ''}</td>"
                   f"<td>{r['nuclei'] if r['nuclei'] is not None else ''}</td><td>{r['fhir_valid']}</td><td>{r['source']}</td></tr>" for r in GR["reports"])
    (G_OUT / "reports" / "index.html").write_text(f"""<!doctype html><html><head><meta charset="utf-8"><title>MMAF-CD example reports</title>
    <style>body{{font-family:Arial;margin:24px}} td,th{{border:1px solid #ddd;padding:3px 8px;font-size:13px}} table{{border-collapse:collapse}}</style></head><body>
    <h2>MMAF-CD v25 — example decision-support reports ({len(GR['reports'])})</h2><p>{_html.escape(DISCLAIMER)}</p>
    <table><tr><th>case</th><th>finding</th><th>category</th><th>p</th><th>accepted</th><th>heatmap check</th><th>outline</th><th>nuclei</th><th>FHIR valid</th><th>source</th></tr>{rows}</table></body></html>""")
    nv = [r["fhir_valid"] for r in GR["reports"]]
    print(f"\n✅ {len(GR['reports'])} reports · FHIR R4B valid: {sum(v is True for v in nv)}/{len(nv)}")
    g_save()

except Exception as _e:
    import traceback; traceback.print_exc()
    print('⚠️ G4 reports failed — skipped:', type(_e).__name__, _e)
    try: g_save()
    except Exception: pass


In [ ]:
try:
    # ════════════════════════════════════════════════════════════════════════════════
    # G5 — F3n merged: official HoVer-Net vs MMAF HoVer-Net, same tiles and same PQ function as Stage F
    #   PanNuke : last 10 % of Part 1–3 concatenated, first 150 tiles (exactly F's pannuke_tiles)
    #   MoNuSeg : all 256-px tiles with nuclei (exactly F's monuseg_tiles with the fixed mask reader)
    # ════════════════════════════════════════════════════════════════════════════════
    MMAF_PQ = {"pannuke": {"PQ": 0.3259, "PQ_std": 0.176, "DQ": 0.4363, "SQ": 0.6798, "n": 150, "pred_per_tile": 19.1, "gt_per_tile": 29.1},
               "monuseg": {"PQ": 0.2805, "PQ_std": 0.170, "DQ": 0.4097, "SQ": 0.6115, "n": 338, "pred_per_tile": 22.1, "gt_per_tile": 42.2}}
    def _compute_panoptic_quality(inst_pred, inst_gt, iou_threshold=0.5):
        pred_ids = set(np.unique(inst_pred)) - {0}; gt_ids = set(np.unique(inst_gt)) - {0}
        matched_iou = []; matched_gt = set()
        for p_id in pred_ids:
            p_mask = inst_pred == p_id; best_iou = 0.; best_g = None
            for g_id in gt_ids:
                if g_id in matched_gt: continue
                g_mask = inst_gt == g_id; inter = float((p_mask & g_mask).sum()); union = float((p_mask | g_mask).sum())
                iou = inter / (union + 1e-6)
                if iou > best_iou: best_iou = iou; best_g = g_id
            if best_iou >= iou_threshold: matched_iou.append(best_iou); matched_gt.add(best_g)
        TP = len(matched_iou); FP = len(pred_ids) - TP; FN = len(gt_ids) - TP
        SQ = float(np.mean(matched_iou)) if matched_iou else 0.; DQ = TP / (TP + 0.5 * FP + 0.5 * FN + 1e-6)
        return {"PQ": round(DQ * SQ, 4), "DQ": round(DQ, 4), "SQ": round(SQ, 4)}

    def _inst_from_mask(m):
        m = np.asarray(m)
        if m.ndim == 3:
            if m.shape[2] >= 3 and not (np.array_equal(m[..., 0], m[..., 1]) and np.array_equal(m[..., 0], m[..., 2])):
                m = m[..., :3].astype(np.int64); m = m[..., 0] * 65536 + m[..., 1] * 256 + m[..., 2]
            else: m = m[..., 0]
        vals, cnt = np.unique(m, return_counts=True); bg = vals[cnt.argmax()]
        if len(vals) == 1: return np.zeros(m.shape, np.int32)
        if m.dtype != np.int64 or len(vals) <= 2:
            keep = vals[cnt >= max(1, 0.001 * m.size)]
            far = keep[np.abs(keep.astype(np.float64) - bg).argmax()] if len(keep) > 1 else vals[np.abs(vals.astype(np.float64) - bg).argmax()]
            d = np.abs(m.astype(np.float64) - bg) / max(abs(float(far) - float(bg)), 1e-6)
            if len(vals) <= 2 or (d[d >= 0.25] > 0.75).mean() >= 0.60:
                return cv2.connectedComponents((d > 0.5).astype(np.uint8), connectivity=8)[1].astype(np.int32)
        _, inv = np.unique(m, return_inverse=True)
        return np.where(m != bg, inv.reshape(m.shape).astype(np.int32) + 1, 0).astype(np.int32)

    def pannuke_tiles(n=150):
        base = find_slug("cancer-inst-segmentation-and-classification")
        if not base: print("   PanNuke not attached"); return None
        base = Path(base); parts = []
        for d in sorted(base.glob("Part *")):
            if (d / "Images" / "images.npy").exists() and (d / "Masks" / "masks.npy").exists(): parts.append((d / "Images" / "images.npy", d / "Masks" / "masks.npy"))
        if not parts:
            for ip in sorted(base.rglob("images.npy")):
                mp = ip.parent / "masks.npy"
                if not mp.exists():
                    sib = ip.parent.parent / "Masks" / "masks.npy"; mp = sib if sib.exists() else None
                if mp is not None: parts.append((ip, mp))
        if not parts: print("   PanNuke .npy not found"); return None
        I = [np.load(str(ip), mmap_mode="r") for ip, _ in parts]; M = [np.load(str(mp), mmap_mode="r") for _, mp in parts]
        N = sum(len(a) for a in I); n_test = max(20, N // 10); want = list(range(N - n_test, min(N, N - n_test + n)))
        offs = np.cumsum([0] + [len(a) for a in I]); imgs, gts = [], []
        for gi in want:
            k = int(np.searchsorted(offs, gi, side="right") - 1); li = gi - offs[k]
            im = np.clip(np.asarray(I[k][li]), 0, 255).astype(np.uint8); mk = np.asarray(M[k][li]).astype(np.float32)
            if mk.shape[0] in (5, 6) and mk.ndim == 3 and mk.shape[-1] not in (5, 6): mk = mk.transpose(1, 2, 0)
            g = np.zeros(mk.shape[:2], np.int32); off = 0
            for c in range(5):
                for iid in np.unique(mk[:, :, c]):
                    if iid == 0: continue
                    off += 1; g[mk[:, :, c] == iid] = off
            imgs.append(im); gts.append(g)
        print(f"   PanNuke: {len(parts)} parts, N={N}, test tiles {len(imgs)} (indices {want[0]}–{want[-1]})"); return imgs, gts

    def monuseg_tiles():
        mon = find_slug("monuseg2018")
        if not mon: print("   MoNuSeg not attached"); return None
        ims, msk = {}, {}
        for p in list_images(mon):
            r = Path(p).relative_to(mon).as_posix().lower(); parts = r.split("/"); key = (parts[0], Path(p).stem.strip().lower())
            if "masks" in parts: msk[key] = p
            elif "images" in parts: ims[key] = p
        imgs, gts = [], []
        for k in [k for k in ims if k in msk]:
            im = np.array(Image.open(ims[k]).convert("RGB")); g = _inst_from_mask(np.array(Image.open(msk[k])))
            if g.shape != im.shape[:2]: g = cv2.resize(g, im.shape[1::-1], interpolation=cv2.INTER_NEAREST)
            for y0 in range(0, im.shape[0] - 255, 256):
                for x0 in range(0, im.shape[1] - 255, 256):
                    gg = g[y0:y0 + 256, x0:x0 + 256]
                    if (gg > 0).sum() > 0: imgs.append(im[y0:y0 + 256, x0:x0 + 256]); gts.append(gg)
        print(f"   MoNuSeg: {len(ims)} images, {len(msk)} masks → {len(imgs)} tiles"); return imgs, gts

    if HV is None: print("⚠️ official HoVer-Net unavailable — nuclei comparison skipped")
    else:
        for name, fn in [("pannuke", pannuke_tiles), ("monuseg", monuseg_tiles)]:
            try:
                T_ = fn()
                if not T_ or not T_[0]: continue
                imgs, gts = T_; t0 = time.time(); preds = [hovernet_tile(im)[0] for im in imgs]
                r = [_compute_panoptic_quality(p, g) for p, g in zip(preds, gts)]
                GR["nuclei_pq"][name] = {"official": {"PQ": float(np.mean([x["PQ"] for x in r])), "PQ_std": float(np.std([x["PQ"] for x in r])),
                                                      "DQ": float(np.mean([x["DQ"] for x in r])), "SQ": float(np.mean([x["SQ"] for x in r])), "n": len(r),
                                                      "pred_per_tile": float(np.mean([len(np.unique(p)) - 1 for p in preds])),
                                                      "gt_per_tile": float(np.mean([len(np.unique(g)) - 1 for g in gts])), "seconds": time.time() - t0},
                                         "mmaf": MMAF_PQ[name]}
                o_ = GR["nuclei_pq"][name]["official"]; m_ = MMAF_PQ[name]
                print(f"  {name:<8} official HoVer-Net PQ={o_['PQ']:.4f}±{o_['PQ_std']:.3f} DQ={o_['DQ']:.3f} SQ={o_['SQ']:.3f} n={o_['n']} "
                      f"nuclei/tile {o_['pred_per_tile']:.1f} (GT {o_['gt_per_tile']:.1f})   |   MMAF PQ={m_['PQ']:.4f} (GT/tile in F: {m_['gt_per_tile']})", flush=True)
            except Exception as e:
                import traceback; traceback.print_exc(); print(f"   ⚠️ nuclei {name}: {e}")
            g_save(); free_mem(name)

except Exception as _e:
    import traceback; traceback.print_exc()
    print('⚠️ G5 nuclei failed — skipped:', type(_e).__name__, _e)
    try: g_save()
    except Exception: pass


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# G6 — tables + package
# ════════════════════════════════════════════════════════════════════════════════
import shutil
def tex(s): return str(s).replace("_", r"\_").replace("&", r"\&")
def fmt(v, nd=3): return "--" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{v:.{nd}f}"
if GR["eval"]:
    L = [r"\begin{tabular}{@{}lrccccccc@{}}", r"\toprule",
         r"Organ & $n$ & Review & Acc.\ all & Acc.\ accepted & Silent err. & Errors caught & Set cov. & Map consistent \\", r"\midrule"]
    for o, r in GR["eval"].items():
        L.append(f"{tex(o)} & {r['n']} & {fmt(r['review_rate'])} & {fmt(r['acc_all'])} & {fmt(r['acc_accepted'])} & {fmt(r['silent_error'])} & "
                 f"{fmt(r['errors_caught'])} & {fmt(r['set_coverage'])} & {fmt(r['heatmap_check'].get('consistent'))} \\\\")
    if "eval_pooled" in GR:
        q = GR["eval_pooled"]; L += [r"\midrule", f"pooled & -- & {fmt(q['review_rate'])} & {fmt(q['acc_all'])} & -- & {fmt(q['silent_error'])} & -- & {fmt(q['set_coverage'])} & -- \\\\"]
    L += [r"\bottomrule", r"\end{tabular}"]; (G_OUT / "tab_reliability.tex").write_text("\n".join(L))
if GR["nuclei_pq"]:
    L = [r"\begin{tabular}{@{}llcccccc@{}}", r"\toprule", r"Set & Model & PQ & DQ & SQ & $n$ & nuclei/tile & GT/tile \\", r"\midrule"]
    for s, v in GR["nuclei_pq"].items():
        for lab, k in [("MMAF HoVer-Net (ours)", "mmaf"), ("HoVer-Net (official, PanNuke weights)", "official")]:
            x = v[k]; L.append(f"{s} & {lab} & {x['PQ']:.3f}$\\pm${x['PQ_std']:.3f} & {x['DQ']:.3f} & {x['SQ']:.3f} & {x['n']} & {x['pred_per_tile']:.1f} & {x['gt_per_tile']:.1f} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]; (G_OUT / "tab_nuclei.tex").write_text("\n".join(L))
GR["meta"]["elapsed_h"] = e_elapsed_h(); g_save()
z = shutil.make_archive("/kaggle/working/mmaf_v25_G", "zip", G_OUT)
print("✅ tables:", sorted(p.name for p in G_OUT.glob("*.tex")), f"\n✅ {z}  ({Path(z).stat().st_size/1e6:.1f} MB) — elapsed {e_elapsed_h():.2f} h")
